# Download images

In [ ]:
var DRIVE_FOLDER = "S2_ML_Dataset_V5";
var SCALE = 10;
var START_DATE = "2023-05-01";
var END_DATE = "2023-08-28";
var locations = [
  { name: "CairoUniv", lon: 31.20909, lat: 30.02633 },
  { name: 'CairoDowntown', lon: 31.2357, lat: 30.0444},
  { name: "NaserCity", lon: 31.33, lat: 30.06 },
  { name: "IconicTower", lon: 31.7018, lat: 30.0109 },
  { name: "AlexandriaPort", lon: 29.8954, lat: 31.2001 },
  { name: "MansouraCity", lon: 31.3667, lat: 31.0333 },
  { name: "HelwanIndustrial", lon: 31.3342, lat: 29.85 },
  { name: "TantaCity", lon: 30.7987, lat: 30.7865 },
  { name: "PortSaidCity", lon: 32.2833, lat: 31.2667 },
  { name: "ZagazigCity", lon: 31.502, lat: 30.5877 },
  { name: "SuezCity", lon: 32.5498, lat: 29.9668 },
  { name: "SiwaOasis", lon: 25.5195, lat: 29.2032 },
  { name: "HawaraFayoum", lon: 30.8986, lat: 29.2747 },
  { name: "EasternDesert", lon: 32.5, lat: 27.5 },
  { name: "WhiteDesert", lon: 28.3, lat: 27.95 },
  { name: "DakhlaOasis", lon: 28.99, lat: 25.49 },
  { name: "GizaPlateau", lon: 31.1342, lat: 29.9792 },
  { name: "QattaraDepression", lon: 27.0, lat: 29.5 },
  { name: "WadiRum", lon: 33.5, lat: 25.0 },
  { name: "SinaiDesert", lon: 33.8, lat: 29.5 },
  { name: "LibyanBorder", lon: 24.5, lat: 29.0 },
  { name: "NileDelta", lon: 31.1, lat: 30.8 },
  { name: "FayoumFarms", lon: 30.84, lat: 29.31 },
  { name: "KarnakLuxor", lon: 32.6583, lat: 25.718 },
  { name: "AssiutFarms", lon: 31.18, lat: 27.18 },
  { name: "ZagazigFarms", lon: 31.5, lat: 30.58 },
  { name: "BeheiraNileDelta", lon: 30.35, lat: 30.85 },
  { name: "MiniaFarms", lon: 30.75, lat: 28.1 },
  { name: "SohagFarms", lon: 31.69, lat: 26.55 },
  { name: "QenaFarms", lon: 32.72, lat: 26.16 },
  { name: "DamietaFarms", lon: 31.8, lat: 31.42 },
  { name: "SaddElAly", lon: 32.88098 , lat: 23.95591 },
  { name: "NasserLake", lon: 32.87, lat: 23.0 },
  { name: "ManzalaLake", lon: 32.0, lat: 31.2 },
  { name: "SuezCanal", lon: 32.55, lat: 30.5 },
  { name: "NileCairo", lon: 31.224, lat: 30.06 },
  { name: "MariutLake", lon: 29.75, lat: 31.15 },
  { name: "QarunLake", lon: 30.55, lat: 29.48 },
  { name: 'FarafraOasis',lon: 27.9700,  lat: 27.0600},  
  { name: 'BaharyaOasis', lon: 28.8391,  lat: 28.410},  
  { name: 'KhargaOasis', lon: 30.5026,  lat: 25.44751},  
  { name: 'WadiElGemal', lon: 34.759845,  lat:24.5387},  
  { name: 'WadiNatrun', lon: 30.4116,  lat: 30.3992},  
  { name: 'AbuSimbelDesert',lon: 31.6257,  lat: 22.3372},  
  { name: 'RedSeaHurghada', lon: 33.8500,  lat: 27.1800},  
  { name: 'GulfAqaba', lon: 34.9350,  lat: 29.5000},  
  { name: 'GulfSuez', lon: 32.524,  lat: 29.431},     
  { name: 'TimsahLake', lon: 32.3000,  lat: 30.5800},  
  { name: 'GreatBitterLake', lon: 32.3500,  lat: 30.3300},  
  { name: 'BardawilLake', lon: 33.160,  lat: 31.1540}, 
];

function maskClouds(image) {
  var scl = image.select("SCL");
  var mask = scl.eq(4).or(scl.eq(5)).or(scl.eq(6));
  return image.updateMask(mask);
}

function getESRIMask(region) {
  return ee.ImageCollection(
      'projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS')
    .filterBounds(region)
    .filter(ee.Filter.calendarRange(2023, 2023, 'year'))  // ← keep 2023
    .first()
    .clip(region)
    .select('b1')
    .remap([1, 2, 4, 5, 7, 8, 9, 10, 11],
           [3, 1, 1, 1, 4, 2, 0, 0, 1])
    .rename('mask')
    .toByte();
}

// function getDWMask(region) {
//   return ee
//     .ImageCollection("GOOGLE/DYNAMICWORLD/V1")
//     .filterBounds(region)
//     .filterDate(START_DATE, END_DATE)
//     .mode()
//     .clip(region)
//     .select("label")
//     .remap([0, 1, 2, 3, 4, 5, 6, 7, 8], [3, 1, 1, 1, 1, 1, 4, 2, 0])
//     .rename("mask")
//     .toByte();
// }

// MAIN LOOP
locations.forEach(function (loc) {
  var region = ee.Geometry.Point([loc.lon, loc.lat]).buffer(1500).bounds();

  var s2 = ee
    .ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(region)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .map(maskClouds)
    .select([
      "B1",
      "B2",
      "B3",
      "B4",
      "B5",
      "B6",
      "B7",
      "B8",
      "B8A",
      "B9",
      "B11",
      "B12",
    ])
    .median()
    .clip(region);

  var rgb = s2
    .select(["B4", "B3", "B2"])
    .divide(3000)
    .multiply(255)
    .clamp(0, 255)
    .toByte();

  var esriMask = getESRIMask(region);
  // var dwMask = getDWMask(region);

  Export.image.toDrive({
    image: s2,
    description: loc.name + "_12band",
    folder: DRIVE_FOLDER,
    scale: SCALE,
    region: region,
    maxPixels: 1e9,
    fileFormat: "GeoTIFF",
  });

  Export.image.toDrive({
    image: esriMask,
    description: loc.name + "_mask_ESRI",
    folder: DRIVE_FOLDER,
    scale: SCALE,
    region: region,
    maxPixels: 1e9,
    fileFormat: "GeoTIFF",
  });

  // Export.image.toDrive({
  //   image: dwMask,
  //   description: loc.name + "_mask_DW",
  //   folder: DRIVE_FOLDER,
  //   scale: SCALE,
  //   region: region,
  //   maxPixels: 1e9,
  //   fileFormat: "GeoTIFF",
  // });

  Export.image.toDrive({
    image: rgb,
    description: loc.name + "_RGB",
    folder: DRIVE_FOLDER,
    scale: SCALE,
    region: region,
    maxPixels: 1e9,
    fileFormat: "GeoTIFF",
  });

  print("Tasks ready: " + loc.name);
});

print("All tasks created! Go to Tasks tab and click Run on each.");


# traverse images

In [ ]:
// 1. YOUR DATA LIST (Truncated example - keep your full 1000 list here)
var locations = [
  { name: "CairoUniv", lon: 31.20909, lat: 30.02633 },
{ name: 'sand_0', lon: 30.80061, lat: 29.50690 },
{ name: 'sand_1', lon: 30.78859, lat: 29.50868 },
];

// 2. NAVIGATION STATE
var currentIndex = 0;

// 3. UI PANEL SETUP
var panel = ui.Panel({
  style: {width: '200px', position: 'bottom-left', padding: '8px'}
});

var title = ui.Label('Location Viewer', {fontWeight: 'bold', fontSize: '16px'});
var label = ui.Label('Showing: ' + locations[currentIndex].name);

// 4. FUNCTION TO UPDATE MAP
function goToLocation(index) {
  var loc = locations[index];
  var point = ee.Geometry.Point([loc.lon, loc.lat]);
  
  Map.centerObject(point, 15.4); // Zoom level 13 is good for landcover
  label.setValue('Index: ' + index + ' | ' + loc.name);
  
  // Optional: Add a temporary marker to show exactly where the point is
  var layers = Map.layers();
  if (layers.length > 3) { Map.remove(layers.get(layers.length - 1)); }
  Map.addLayer(point, {color: 'red'}, 'Current Point');
}

// 5. BUTTONS
var nextBtn = ui.Button({
  label: 'Next ❯',
  onClick: function() {
    currentIndex = (currentIndex + 1) % locations.length;
    goToLocation(currentIndex);
  }
});

var prevBtn = ui.Button({
  label: '❮ Prev',
  onClick: function() {
    currentIndex = (currentIndex - 1 + locations.length) % locations.length;
    goToLocation(currentIndex);
  }
});

// 6. ASSEMBLE AND SHOW
panel.add(title);
panel.add(label);
panel.add(ui.Panel([prevBtn, nextBtn], ui.Panel.Layout.flow('horizontal')));
Map.add(panel);

// Initialize first location
goToLocation(0);


# get locs manually

In [ ]:
// --- UI SETUP ---
var savedLocations = [];
var counter = 0;

var panel = ui.Panel({
  style: {width: '250px', position: 'bottom-left', padding: '10px'}
});

var title = ui.Label('Dataset Collector', {fontWeight: 'bold', fontSize: '18px'});
var instructions = ui.Label('1. Navigate freely\n2. Click "Fix Zoom"\n3. Click "Add Location"', {whiteSpace: 'pre'});

var labelInput = ui.Textbox({
  placeholder: 'Enter class (e.g. sand)',
  value: 'sand'
});

var countLabel = ui.Label('Saved: 0');

// --- BUTTON FUNCTIONS ---

// NEW: Snap back to your specific training zoom level
var fixZoomBtn = ui.Button({
  label: '🔍 Fix Zoom (15.4)',
  onClick: function() {
    Map.setZoom(15.4);
  },
  style: {stretch: 'horizontal', color: 'blue'}
});

var addBtn = ui.Button({
  label: '📍 Add Location',
  onClick: function() {
    var center = Map.getCenter().coordinates().getInfo();
    var className = labelInput.getValue();
    
    // center[0] is Lon, center[1] is Lat
    var entry = "{ name: '" + className + "_" + counter + "', lon: " + center[0].toFixed(5) + ", lat: " + center[1].toFixed(5) + " },";
    
    savedLocations.push(entry);
    counter++;
    
    countLabel.setValue('Saved: ' + counter);
    print('Added: ' + entry);
  },
  style: {stretch: 'horizontal'}
});

var finishBtn = ui.Button({
  label: '✅ Finish & Show List',
  onClick: function() {
    print('--- COPY DATA BELOW ---');
    savedLocations.forEach(function(loc) {
      print(loc);
    });
    print('--- END OF DATA ---');
    alert('List printed to Console! Copy it from there.');
  },
  style: {stretch: 'horizontal'}
});

// --- ASSEMBLE PANEL ---
panel.add(title);
panel.add(instructions);
panel.add(ui.Label('Class Label:'));
panel.add(labelInput);
panel.add(fixZoomBtn); // Added to panel
panel.add(addBtn);
panel.add(countLabel);
panel.add(finishBtn);

Map.add(panel);
Map.setControlVisibility({zoomControl: true});
Map.setCenter(31.2357, 30.0444, 15.4); // Initial view


In [ ]:
# { "name": "green_0", "lon": 31.01946, "lat": 30.80033 },
# { "name": "green_1", "lon": 30.9837, "lat": 30.82556 },
# { "name": "green_2", "lon": 31.00227, "lat": 30.83301 },
# { "name": "green_3", "lon": 30.95219, "lat": 30.76905 },
# { "name": "green_4", "lon": 31.02574, "lat": 30.79923 },
# { "name": "green_5", "lon": 30.95122, "lat": 30.78616 },
# { "name": "green_6", "lon": 31.01609, "lat": 30.81476 },
# { "name": "green_7", "lon": 31.03524, "lat": 30.81971 },
# { "name": "green_8", "lon": 30.96416, "lat": 30.76827 },
# { "name": "green_9", "lon": 31.00957, "lat": 30.80651 },
# { "name": "green_10", "lon": 30.97441, "lat": 30.80859 },
# { "name": "green_11", "lon": 31.00127, "lat": 30.80501 },
# { "name": "green_12", "lon": 30.97412, "lat": 30.82768 },
# { "name": "green_13", "lon": 31.00439, "lat": 30.76513 },
# { "name": "green_14", "lon": 31.00675, "lat": 30.81833 },
# { "name": "green_15", "lon": 31.03577, "lat": 30.76938 },
# { "name": "green_16", "lon": 30.95395, "lat": 30.80961 },
# { "name": "green_17", "lon": 30.96735, "lat": 30.8006 },
# { "name": "green_18", "lon": 31.0101, "lat": 30.84396 },
# { "name": "green_19", "lon": 31.04895, "lat": 30.79686 },
# { "name": "green_20", "lon": 30.95923, "lat": 30.79019 },
# { "name": "green_21", "lon": 30.9771, "lat": 30.83217 },
# { "name": "green_22", "lon": 30.95907, "lat": 30.7524 },
# { "name": "green_23", "lon": 31.03027, "lat": 30.80459 },
# { "name": "green_24", "lon": 30.97179, "lat": 30.79361 },
# { "name": "green_25", "lon": 30.95946, "lat": 30.79775 },
# { "name": "green_26", "lon": 31.03395, "lat": 30.80118 },
# { "name": "green_27", "lon": 31.03726, "lat": 30.7769 },
# { "name": "green_28", "lon": 30.99607, "lat": 30.76282 },
# { "name": "green_29", "lon": 30.95731, "lat": 30.81563 },
# { "name": "green_30", "lon": 31.03632, "lat": 30.78809 },
# { "name": "green_31", "lon": 31.04712, "lat": 30.7564 },
# { "name": "green_32", "lon": 31.04753, "lat": 30.8141 },
# { "name": "green_33", "lon": 31.02121, "lat": 30.75097 },
# { "name": "green_34", "lon": 31.04814, "lat": 30.84621 },
# { "name": "green_35", "lon": 30.95183, "lat": 30.77661 },
# { "name": "green_36", "lon": 30.65942, "lat": 28.0812 },
# { "name": "green_37", "lon": 30.68055, "lat": 28.05804 },
# { "name": "green_38", "lon": 30.66348, "lat": 28.07741 },
# { "name": "green_39", "lon": 30.68713, "lat": 28.08971 },
# { "name": "green_40", "lon": 30.73422, "lat": 28.069 },
# { "name": "green_41", "lon": 30.69291, "lat": 28.11495 },
# { "name": "green_42", "lon": 30.68083, "lat": 28.13596 },
# { "name": "green_43", "lon": 30.65885, "lat": 28.13508 },
# { "name": "green_44", "lon": 30.69458, "lat": 28.06721 },
# { "name": "green_45", "lon": 30.70401, "lat": 28.07347 },
# { "name": "green_46", "lon": 30.74083, "lat": 28.1049 },
# { "name": "green_47", "lon": 30.73941, "lat": 28.07802 },
# { "name": "green_48", "lon": 30.68921, "lat": 28.12227 },
# { "name": "green_49", "lon": 30.70494, "lat": 28.13965 },
# { "name": "green_50", "lon": 30.68647, "lat": 28.06577 },
# { "name": "green_51", "lon": 30.66258, "lat": 28.0937 },
# { "name": "green_52", "lon": 30.74781, "lat": 28.07369 },
# { "name": "green_53", "lon": 30.70942, "lat": 28.14242 },
# { "name": "green_54", "lon": 30.72329, "lat": 28.08545 },
# { "name": "green_55", "lon": 30.67182, "lat": 28.06958 },
# { "name": "green_56", "lon": 30.71567, "lat": 28.1356 },
# { "name": "green_57", "lon": 30.66046, "lat": 28.11789 },
# { "name": "green_58", "lon": 30.72924, "lat": 28.08845 },
# { "name": "green_59", "lon": 30.67241, "lat": 28.0608 },
# { "name": "green_60", "lon": 30.70154, "lat": 28.14762 },
# { "name": "green_61", "lon": 30.70115, "lat": 28.11958 },
# { "name": "green_62", "lon": 30.72388, "lat": 28.11627 },
# { "name": "green_63", "lon": 30.70538, "lat": 28.13159 },
# { "name": "green_64", "lon": 30.72743, "lat": 28.124 },
# { "name": "green_65", "lon": 30.65312, "lat": 28.12894 },
# { "name": "green_66", "lon": 30.71966, "lat": 28.08245 },
# { "name": "green_67", "lon": 30.71753, "lat": 28.10135 },
# { "name": "green_68", "lon": 30.71192, "lat": 28.0854 },
# { "name": "green_69", "lon": 30.69107, "lat": 28.09455 },
# { "name": "green_70", "lon": 30.66579, "lat": 28.1224 },
# { "name": "green_71", "lon": 30.68275, "lat": 28.12938 },
# { "name": "green_72", "lon": 30.74924, "lat": 28.14038 },
# { "name": "green_73", "lon": 30.71431, "lat": 28.05482 },
# { "name": "green_74", "lon": 31.78131, "lat": 30.50685 },
# { "name": "green_75", "lon": 31.79751, "lat": 30.45013 },
# { "name": "green_76", "lon": 31.83025, "lat": 30.51059 },
# { "name": "green_77", "lon": 31.83754, "lat": 30.50189 },
# { "name": "green_78", "lon": 31.80882, "lat": 30.5163 },
# { "name": "green_79", "lon": 31.83685, "lat": 30.46137 },
# { "name": "green_80", "lon": 31.80868, "lat": 30.50602 },
# { "name": "green_81", "lon": 31.7831, "lat": 30.49782 },
# { "name": "green_82", "lon": 31.80357, "lat": 30.5462 },
# { "name": "green_83", "lon": 31.75127, "lat": 30.51531 },
# { "name": "green_84", "lon": 31.77261, "lat": 30.48398 },
# { "name": "green_85", "lon": 31.78286, "lat": 30.48929 },
# { "name": "green_86", "lon": 31.75562, "lat": 30.52233 },
# { "name": "green_87", "lon": 31.82283, "lat": 30.46895 },
# { "name": "green_88", "lon": 31.84717, "lat": 30.52379 },
# { "name": "green_89", "lon": 31.84399, "lat": 30.46956 },
# { "name": "green_90", "lon": 31.77358, "lat": 30.52185 },
# { "name": "green_91", "lon": 31.79185, "lat": 30.47774 },
# { "name": "green_92", "lon": 31.77776, "lat": 30.47337 },
# { "name": "green_93", "lon": 31.75354, "lat": 30.53149 },
# { "name": "green_94", "lon": 31.7957, "lat": 30.49226 },
# { "name": "green_95", "lon": 31.80554, "lat": 30.46615 },
# { "name": "green_96", "lon": 31.79713, "lat": 30.46092 },
# { "name": "green_97", "lon": 31.78939, "lat": 30.51094 },
# { "name": "green_98", "lon": 31.82493, "lat": 30.49823 },
# { "name": "green_99", "lon": 31.7726, "lat": 30.52841 },
# { "name": "green_100", "lon": 31.75287, "lat": 30.47441 },
# { "name": "green_101", "lon": 31.79582, "lat": 30.54322 },
# { "name": "green_102", "lon": 31.82375, "lat": 30.4804 },
# { "name": "green_103", "lon": 31.76842, "lat": 30.54716 },
# { "name": "green_104", "lon": 31.76836, "lat": 30.51043 },
# { "name": "green_105", "lon": 31.75189, "lat": 30.5007 },
# { "name": "green_106", "lon": 31.83507, "lat": 30.45305 },
# { "name": "green_107", "lon": 31.7994, "lat": 30.524 },
# { "name": "green_108", "lon": 31.76165, "lat": 30.52761 },
# { "name": "green_109", "lon": 31.77091, "lat": 30.50006 },
# { "name": "green_110", "lon": 31.82485, "lat": 30.45004 },
# { "name": "green_111", "lon": 31.84919, "lat": 30.48396 },
# { "name": "green_112", "lon": 28.97066, "lat": 25.52417 },
# { "name": "green_113", "lon": 29.01888, "lat": 25.54772 },
# { "name": "green_114", "lon": 28.96656, "lat": 25.54887 },
# { "name": "green_115", "lon": 28.97565, "lat": 25.47586 },
# { "name": "green_116", "lon": 29.03131, "lat": 25.51661 },
# { "name": "green_117", "lon": 28.99063, "lat": 25.45919 },
# { "name": "green_118", "lon": 28.9523, "lat": 25.4717 },
# { "name": "green_119", "lon": 28.96448, "lat": 25.50375 },
# { "name": "green_120", "lon": 28.9526, "lat": 25.53551 },
# { "name": "green_121", "lon": 29.02478, "lat": 25.48656 },
# { "name": "green_122", "lon": 28.96097, "lat": 25.49841 },
# { "name": "green_123", "lon": 28.96963, "lat": 25.45577 },
# { "name": "green_124", "lon": 29.03892, "lat": 25.53964 },
# { "name": "green_125", "lon": 28.97251, "lat": 25.48017 },
# { "name": "green_126", "lon": 29.00624, "lat": 25.54128 },
# { "name": "green_127", "lon": 28.98704, "lat": 25.45364 },
# { "name": "green_128", "lon": 28.96944, "lat": 25.49205 },
# { "name": "green_129", "lon": 29.01118, "lat": 25.47896 },
# { "name": "green_130", "lon": 29.02292, "lat": 25.47992 },
# { "name": "green_131", "lon": 28.95777, "lat": 25.47158 },
# { "name": "green_132", "lon": 28.9748, "lat": 25.52717 },
# { "name": "green_133", "lon": 28.96533, "lat": 25.49457 },
# { "name": "green_134", "lon": 29.01283, "lat": 25.50146 },
# { "name": "green_135", "lon": 29.01881, "lat": 25.53354 },
# { "name": "green_136", "lon": 29.02733, "lat": 25.49661 },
# { "name": "green_137", "lon": 28.99105, "lat": 25.52202 },
# { "name": "green_138", "lon": 28.96828, "lat": 25.47401 },
# { "name": "green_139", "lon": 28.95177, "lat": 25.51748 },
# { "name": "green_140", "lon": 28.98238, "lat": 25.45842 },

In [ ]:
# { name: 'water_0', lon: 30.48048, lat: 29.42557 },
# { name: 'water_1', lon: 30.53815, lat: 29.42364 },
# { name: 'water_2', lon: 30.58248, lat: 29.4264 },
# { name: 'water_3', lon: 30.62548, lat: 29.43794 },
# { name: 'water_4', lon: 30.64757, lat: 29.45137 },
# { name: 'water_5', lon: 30.66767, lat: 29.47 },
# { name: 'water_6', lon: 30.70649, lat: 29.48062 },
# { name: 'water_7', lon: 30.72713, lat: 29.49512 },
# { name: 'water_8', lon: 30.7594, lat: 29.47923 },
# { name: 'water_9', lon: 30.7833, lat: 29.48143 },
# { name: 'water_10', lon: 30.80517, lat: 29.49137 },
# { name: 'water_11', lon: 30.46257, lat: 29.24235 },
# { name: 'water_12', lon: 30.4655, lat: 29.27591 },
# { name: 'water_13', lon: 30.50604, lat: 29.25248 },
# { name: 'water_14', lon: 30.50776, lat: 29.23012 },
# { name: 'water_15', lon: 30.49164, lat: 29.20533 },
# { name: 'water_16', lon: 30.41227, lat: 29.17657 },
# { name: 'water_17', lon: 30.42035, lat: 29.20065 },
# { name: 'water_18', lon: 30.39459, lat: 29.19559 },
# { name: 'water_19', lon: 25.4876, lat: 29.21989 },
# { name: 'water_20', lon: 25.44759, lat: 29.2213 },
# { name: 'water_21', lon: 25.45258, lat: 29.20064 },
# { name: 'water_22', lon: 25.32915, lat: 29.23271 },
# { name: 'water_23', lon: 25.35518, lat: 29.34558 },
# { name: 'water_24', lon: 25.75142, lat: 29.16313 },
# { name: 'water_25', lon: 26.00854, lat: 29.28526 },
# { name: 'water_26', lon: 26.5352, lat: 29.60777 },
# { name: 'water_27', lon: 19.80753, lat: 29.76988 },
# { name: 'water_28', lon: 19.78742, lat: 29.78924 },
# { name: 'water_29', lon: 19.77307, lat: 29.81038 },
# { name: 'water_30', lon: 19.75754, lat: 29.82176 },
# { name: 'water_31', lon: 19.73496, lat: 29.84281 },
# { name: 'water_32', lon: 19.70717, lat: 29.8527 },
# { name: 'water_33', lon: 19.67561, lat: 29.87453 },
# { name: 'water_34', lon: 19.63655, lat: 29.8892 },
# { name: 'water_35', lon: 19.64704, lat: 29.8686 },
# { name: 'water_36', lon: 19.6656, lat: 29.84498 },
# { name: 'water_37', lon: 19.23048, lat: 29.98327 },
# { name: 'water_38', lon: 19.22751, lat: 29.96323 },
# { name: 'water_39', lon: 19.15623, lat: 29.96846 },
# { name: 'water_40', lon: 19.1342, lat: 29.9908 },
# { name: 'water_41', lon: 19.11347, lat: 29.99736 },
# { name: 'water_42', lon: 19.09151, lat: 30.00165 },
# { name: 'water_43', lon: 19.06135, lat: 30.06856 },
# { name: 'water_44', lon: 19.05079, lat: 30.09868 },
# { name: 'water_45', lon: 19.03918, lat: 30.27818 },
# { name: 'water_46', lon: 19.06082, lat: 30.27407 },
# { name: 'water_47', lon: 19.09938, lat: 30.2813 },
# { name: 'water_48', lon: 19.13202, lat: 30.27918 },
# { name: 'water_49', lon: 19.15147, lat: 30.27732 },
# { name: 'water_50', lon: 19.21098, lat: 30.27831 },
# { name: 'water_51', lon: 19.24802, lat: 30.28946 },
# { name: 'water_52', lon: 19.29331, lat: 30.29664 },
# { name: 'water_53', lon: 19.31814, lat: 30.30613 },
# { name: 'water_54', lon: 19.33744, lat: 30.31544 },
# { name: 'water_55', lon: 19.36701, lat: 30.32471 },
# { name: 'water_56', lon: 19.39667, lat: 30.33599 },
# { name: 'water_57', lon: 19.41993, lat: 30.35066 },
# { name: 'water_58', lon: 19.44285, lat: 30.3598 },
# { name: 'water_59', lon: 19.47351, lat: 30.37662 },
# { name: 'water_60', lon: 19.50376, lat: 30.39445 },
# { name: 'water_61', lon: 19.52718, lat: 30.40562 },
# { name: 'water_62', lon: 19.59066, lat: 30.42337 },
# { name: 'water_63', lon: 19.61874, lat: 30.42864 },
# { name: 'water_64', lon: 19.64, lat: 30.45122 },
# { name: 'water_65', lon: 19.67776, lat: 30.46841 },
# { name: 'water_66', lon: 19.72883, lat: 30.51319 },
# { name: 'water_67', lon: 19.7613, lat: 30.5222 },
# { name: 'water_68', lon: 19.77778, lat: 30.54385 },
# { name: 'water_69', lon: 19.78897, lat: 30.56123 },
# { name: 'water_70', lon: 19.80185, lat: 30.57677 },
# { name: 'water_71', lon: 19.81609, lat: 30.59424 },
# { name: 'water_72', lon: 19.8353, lat: 30.61257 },
# { name: 'water_73', lon: 19.8611, lat: 30.62861 },
# { name: 'water_74', lon: 19.87421, lat: 30.64696 },
# { name: 'water_75', lon: 19.89803, lat: 30.66755 },
# { name: 'water_76', lon: 19.92176, lat: 30.69139 },
# { name: 'water_77', lon: 19.94243, lat: 30.73128 },
# { name: 'water_78', lon: 19.96225, lat: 30.75339 },
# { name: 'water_79', lon: 19.98504, lat: 30.78503 },
# { name: 'water_80', lon: 20.02434, lat: 30.81822 },
# { name: 'water_81', lon: 20.04056, lat: 30.84921 },
# { name: 'water_82', lon: 20.06205, lat: 30.89561 },
# { name: 'water_83', lon: 20.08237, lat: 30.92204 },
# { name: 'water_84', lon: 20.09415, lat: 30.95441 },
# { name: 'water_85', lon: 20.11397, lat: 30.98443 },
# { name: 'water_86', lon: 20.12965, lat: 31.0085 },
# { name: 'water_87', lon: 20.1387, lat: 31.03595 },
# { name: 'water_88', lon: 20.15773, lat: 31.08815 },
# { name: 'water_89', lon: 20.1589, lat: 31.11735 },
# { name: 'water_90', lon: 20.15963, lat: 31.13906 },
# { name: 'water_91', lon: 20.15681, lat: 31.17069 },
# { name: 'water_92', lon: 20.14247, lat: 31.22184 },
# { name: 'water_93', lon: 20.12305, lat: 31.25315 },
# { name: 'water_94', lon: 20.10519, lat: 31.28631 },
# { name: 'water_95', lon: 20.08587, lat: 31.31894 },
# { name: 'water_96', lon: 20.05887, lat: 31.34711 },
# { name: 'water_97', lon: 20.04175, lat: 31.35722 },
# { name: 'water_98', lon: 20.02664, lat: 31.37412 },
# { name: 'water_99', lon: 20.01636, lat: 31.3916 },
# { name: 'water_100', lon: 20.00594, lat: 31.41918 },
# { name: 'water_101', lon: 20.00298, lat: 31.45063 },
# { name: 'water_102', lon: 19.97233, lat: 31.53172 },
# { name: 'water_103', lon: 19.96561, lat: 31.55134 },
# { name: 'water_104', lon: 19.96053, lat: 31.57048 },
# { name: 'water_105', lon: 19.95245, lat: 31.5912 },
# { name: 'water_106', lon: 19.94458, lat: 31.65129 },
# { name: 'water_107', lon: 19.94169, lat: 31.67498 },
# { name: 'water_108', lon: 19.9318, lat: 31.69783 },
# { name: 'water_109', lon: 19.91778, lat: 31.72151 },
# { name: 'water_110', lon: 19.91022, lat: 31.75795 },
# { name: 'water_111', lon: 19.91479, lat: 31.77783 },
# { name: 'water_112', lon: 19.91841, lat: 31.80261 },
# { name: 'water_113', lon: 19.91764, lat: 31.83047 },
# { name: 'water_114', lon: 19.93418, lat: 31.88327 },
# { name: 'water_115', lon: 19.93879, lat: 31.98579 },
# { name: 'water_116', lon: 20.00376, lat: 32.05349 },
# { name: 'water_117', lon: 20.035, lat: 32.12127 },
# { name: 'water_118', lon: 20.05469, lat: 32.13646 },
# { name: 'water_119', lon: 20.08817, lat: 32.17559 },
# { name: 'water_120', lon: 20.10385, lat: 32.19651 },
# { name: 'water_121', lon: 20.14023, lat: 32.21983 },
# { name: 'water_122', lon: 20.15193, lat: 32.24327 },
# { name: 'water_123', lon: 20.17018, lat: 32.25873 },
# { name: 'water_124', lon: 20.23814, lat: 32.3249 },
# { name: 'water_125', lon: 20.27229, lat: 32.34688 },
# { name: 'water_126', lon: 20.37438, lat: 32.43218 },
# { name: 'water_127', lon: 20.39868, lat: 32.44983 },
# { name: 'water_128', lon: 20.4355, lat: 32.4698 },
# { name: 'water_129', lon: 20.45411, lat: 32.48303 },
# { name: 'water_130', lon: 20.48477, lat: 32.50279 },
# { name: 'water_131', lon: 20.52126, lat: 32.52704 },
# { name: 'water_132', lon: 30.78859, lat: 29.50868 },
# { name: 'water_133', lon: 30.76732, lat: 29.51554 },
# { name: 'water_134', lon: 30.74304, lat: 29.51164 },
# { name: 'water_135', lon: 30.70149, lat: 29.50661 },
# { name: 'water_136', lon: 30.6568, lat: 29.49053 },
# { name: 'water_137', lon: 30.62294, lat: 29.49776 },
# { name: 'water_138', lon: 30.59385, lat: 29.48708 },
# { name: 'water_139', lon: 30.55727, lat: 29.46443 },
# { name: 'water_140', lon: 30.4975, lat: 29.44026 },

In [ ]:
# { name: 'sand_0', lon: 26.99901, lat: 24.97902 },
# { name: 'sand_1', lon: 26.998, lat: 24.95723 },
# { name: 'sand_2', lon: 26.99189, lat: 25.0131 },
# { name: 'sand_3', lon: 26.97482, lat: 24.98258 },
# { name: 'sand_4', lon: 26.95485, lat: 25.04057 },
# { name: 'sand_5', lon: 27.01875, lat: 24.95081 },
# { name: 'sand_6', lon: 26.96884, lat: 25.03838 },
# { name: 'sand_7', lon: 26.98233, lat: 25.02617 },
# { name: 'sand_8', lon: 26.99427, lat: 25.0379 },
# { name: 'sand_9', lon: 27.0294, lat: 24.96465 },
# { name: 'sand_10', lon: 26.95291, lat: 25.02923 },
# { name: 'sand_11', lon: 26.97238, lat: 25.04773 },
# { name: 'sand_12', lon: 27.01878, lat: 25.00403 },
# { name: 'sand_13', lon: 26.99228, lat: 24.99685 },
# { name: 'sand_14', lon: 26.96425, lat: 25.01003 },
# { name: 'sand_15', lon: 26.97863, lat: 24.96333 },
# { name: 'sand_16', lon: 27.0339, lat: 25.03826 },
# { name: 'sand_17', lon: 26.9792, lat: 25.03859 },
# { name: 'sand_18', lon: 27.00695, lat: 25.00063 },
# { name: 'sand_19', lon: 26.97037, lat: 24.99288 },
# { name: 'sand_20', lon: 27.03058, lat: 25.01668 },
# { name: 'sand_21', lon: 27.04458, lat: 25.04186 },
# { name: 'sand_22', lon: 27.03536, lat: 24.97419 },
# { name: 'sand_23', lon: 26.95335, lat: 24.99311 },
# { name: 'sand_24', lon: 28.96696, lat: 28.00904 },
# { name: 'sand_25', lon: 29.02942, lat: 28.01883 },
# { name: 'sand_26', lon: 28.97547, lat: 28.02156 },
# { name: 'sand_27', lon: 29.04525, lat: 27.96149 },
# { name: 'sand_28', lon: 29.00199, lat: 27.97943 },
# { name: 'sand_29', lon: 29.02208, lat: 28.04815 },
# { name: 'sand_30', lon: 28.95723, lat: 28.03186 },
# { name: 'sand_31', lon: 29.00508, lat: 28.0029 },
# { name: 'sand_32', lon: 28.95422, lat: 27.9556 },
# { name: 'sand_33', lon: 29.01039, lat: 27.96547 },
# { name: 'sand_34', lon: 29.03715, lat: 27.97563 },
# { name: 'sand_35', lon: 29.02565, lat: 27.99915 },
# { name: 'sand_36', lon: 29.02208, lat: 27.96496 },
# { name: 'sand_37', lon: 28.98763, lat: 27.97586 },
# { name: 'sand_38', lon: 29.00896, lat: 27.95265 },
# { name: 'sand_39', lon: 28.98889, lat: 27.95519 },
# { name: 'sand_40', lon: 29.0198, lat: 28.0118 },
# { name: 'sand_41', lon: 29.00474, lat: 28.03531 },
# { name: 'sand_42', lon: 28.95774, lat: 27.97254 },
# { name: 'sand_43', lon: 29.04745, lat: 28.03808 },
# { name: 'sand_44', lon: 28.96503, lat: 27.96254 },
# { name: 'sand_45', lon: 29.04691, lat: 28.02335 },
# { name: 'sand_46', lon: 29.01759, lat: 28.03117 },
# { name: 'sand_47', lon: 28.97532, lat: 27.98524 },
# { name: 'sand_48', lon: 29.0378, lat: 27.99123 },
# { name: 'sand_49', lon: 29.03734, lat: 28.00242 },
# { name: 'sand_50', lon: 28.96183, lat: 27.98278 },
# { name: 'sand_51', lon: 28.98933, lat: 28.04966 },
# { name: 'sand_52', lon: 28.96704, lat: 28.04084 },
# { name: 'sand_53', lon: 28.98469, lat: 28.00117 },
# { name: 'sand_54', lon: 28.9557, lat: 28.0133 },
# { name: 'sand_55', lon: 30.96799, lat: 27.47149 },
# { name: 'sand_56', lon: 31.01609, lat: 27.49683 },
# { name: 'sand_57', lon: 30.96066, lat: 27.49207 },
# { name: 'sand_58', lon: 30.96891, lat: 27.51166 },
# { name: 'sand_59', lon: 30.95231, lat: 27.53991 },
# { name: 'sand_60', lon: 30.98661, lat: 27.52097 },
# { name: 'sand_61', lon: 30.97039, lat: 27.45181 },
# { name: 'sand_62', lon: 30.98306, lat: 27.49207 },
# { name: 'sand_63', lon: 31.0206, lat: 27.46025 },
# { name: 'sand_64', lon: 31.02678, lat: 27.53555 },
# { name: 'sand_65', lon: 30.96188, lat: 27.52242 },
# { name: 'sand_66', lon: 31.01335, lat: 27.532 },
# { name: 'sand_67', lon: 31.00452, lat: 27.5376 },
# { name: 'sand_68', lon: 30.96116, lat: 27.53471 },
# { name: 'sand_69', lon: 30.98573, lat: 27.45645 },
# { name: 'sand_70', lon: 31.03715, lat: 27.46026 },
# { name: 'sand_71', lon: 31.0344, lat: 27.50378 },
# { name: 'sand_72', lon: 31.03661, lat: 27.48379 },
# { name: 'sand_73', lon: 31.04284, lat: 27.47188 },
# { name: 'sand_74', lon: 31.02006, lat: 27.50846 },
# { name: 'sand_75', lon: 31.04927, lat: 27.45647 },
# { name: 'sand_76', lon: 30.96285, lat: 27.50257 },
# { name: 'sand_77', lon: 30.95778, lat: 27.47409 },
# { name: 'sand_78', lon: 30.97919, lat: 27.46714 },
# { name: 'sand_79', lon: 30.97439, lat: 27.47942 },
# { name: 'sand_80', lon: 31.01594, lat: 27.45133 },
# { name: 'sand_81', lon: 31.03363, lat: 27.52061 },
# { name: 'sand_82', lon: 31.00477, lat: 27.5012 },
# { name: 'sand_83', lon: 31.00534, lat: 27.45486 },
# { name: 'sand_84', lon: 30.99534, lat: 27.51508 },
# { name: 'sand_85', lon: 30.98594, lat: 27.54193 },
# { name: 'sand_86', lon: 27.46726, lat: 28.95685 },
# { name: 'sand_87', lon: 27.50969, lat: 29.0451 },
# { name: 'sand_88', lon: 27.53307, lat: 28.99221 },
# { name: 'sand_89', lon: 27.5222, lat: 29.0146 },
# { name: 'sand_90', lon: 27.46823, lat: 29.04902 },
# { name: 'sand_91', lon: 27.49809, lat: 29.01738 },
# { name: 'sand_92', lon: 27.45267, lat: 28.97308 },
# { name: 'sand_93', lon: 27.51117, lat: 28.96651 },
# { name: 'sand_94', lon: 27.50219, lat: 28.99534 },
# { name: 'sand_95', lon: 27.48651, lat: 28.96067 },
# { name: 'sand_96', lon: 27.54275, lat: 29.01295 },
# { name: 'sand_97', lon: 27.45523, lat: 29.03538 },
# { name: 'sand_98', lon: 27.45806, lat: 29.04989 },
# { name: 'sand_99', lon: 27.53392, lat: 29.02254 },
# { name: 'sand_100', lon: 27.45682, lat: 29.00128 },
# { name: 'sand_101', lon: 27.53626, lat: 28.97115 },
# { name: 'sand_102', lon: 27.47081, lat: 29.02979 },
# { name: 'sand_103', lon: 27.541, lat: 29.03255 },
# { name: 'sand_104', lon: 27.54764, lat: 28.9564 },
# { name: 'sand_105', lon: 27.48308, lat: 28.98377 },
# { name: 'sand_106', lon: 27.48851, lat: 28.99977 },
# { name: 'sand_107', lon: 27.54264, lat: 28.98118 },
# { name: 'sand_108', lon: 27.48519, lat: 29.03825 },
# { name: 'sand_109', lon: 27.46537, lat: 28.97847 },
# { name: 'sand_110', lon: 27.48813, lat: 29.01186 },
# { name: 'sand_111', lon: 27.49384, lat: 28.95101 },
# { name: 'sand_112', lon: 27.49912, lat: 28.96758 },
# { name: 'sand_113', lon: 27.47352, lat: 28.96682 },
# { name: 'sand_114', lon: 32.04547, lat: 23.48376 },
# { name: 'sand_115', lon: 31.99231, lat: 23.49824 },
# { name: 'sand_116', lon: 32.02352, lat: 23.47608 },
# { name: 'sand_117', lon: 32.03696, lat: 23.45234 },
# { name: 'sand_118', lon: 31.95856, lat: 23.47054 },
# { name: 'sand_119', lon: 31.96933, lat: 23.5407 },
# { name: 'sand_120', lon: 31.95552, lat: 23.50498 },
# { name: 'sand_121', lon: 31.96551, lat: 23.51695 },
# { name: 'sand_122', lon: 31.98902, lat: 23.47944 },
# { name: 'sand_123', lon: 32.01232, lat: 23.52999 },
# { name: 'sand_124', lon: 31.98496, lat: 23.45191 },
# { name: 'sand_125', lon: 31.96911, lat: 23.49556 },
# { name: 'sand_126', lon: 32.00552, lat: 23.48219 },
# { name: 'sand_127', lon: 32.01062, lat: 23.45431 },
# { name: 'sand_128', lon: 32.04651, lat: 23.46468 },
# { name: 'sand_129', lon: 32.04597, lat: 23.54319 },
# { name: 'sand_130', lon: 31.98814, lat: 23.51663 },
# { name: 'sand_131', lon: 32.02738, lat: 23.46119 },
# { name: 'sand_132', lon: 31.95746, lat: 23.48931 },
# { name: 'sand_133', lon: 32.02901, lat: 23.50888 },
# { name: 'sand_134', lon: 31.97914, lat: 23.46487 },
# { name: 'sand_135', lon: 31.95223, lat: 23.54573 },
# { name: 'sand_136', lon: 32.0048, lat: 23.52053 },
# { name: 'sand_137', lon: 32.02739, lat: 23.49665 },
# { name: 'sand_138', lon: 32.0355, lat: 23.53589 },
# { name: 'sand_139', lon: 32.04269, lat: 23.51965 },
# { name: 'sand_140', lon: 31.97764, lat: 23.53492 },

In [ ]:
# { name: "cement_0", lon: 31.20909, lat: 30.02633 },
# { name: "cement_1", lon: 31.74655, lat: 30.02777 },
# { name: "cement_2", lon: 31.65059, lat: 30.01940 },
# { name: "cement_3", lon: 31.66263, lat: 29.96356 },
# { name: "cement_4", lon: 31.74477, lat: 30.02578 },
# { name: "cement_5", lon: 31.70585, lat: 30.02073 },
# { name: "cement_6", lon: 31.72424, lat: 30.00147 },
# { name: "cement_7", lon: 31.67506, lat: 29.99444 },
# { name: "cement_8", lon: 31.68901, lat: 30.03448 },
# { name: "cement_9", lon: 31.70660, lat: 30.03673 },
# { name: "cement_10", lon: 31.67970, lat: 29.96124 },
# { name: "cement_11", lon: 31.68795, lat: 30.02257 },
# { name: "cement_12", lon: 31.67960, lat: 30.03342 },
# { name: "cement_13", lon: 31.71830, lat: 29.98199 },
# { name: "cement_14", lon: 31.73910, lat: 30.00460 },
# { name: "cement_15", lon: 31.68035, lat: 30.02480 },
# { name: "cement_16", lon: 31.69756, lat: 30.04103 },
# { name: "cement_17", lon: 31.67529, lat: 29.95184 },
# { name: "cement_18", lon: 31.71649, lat: 30.01043 },
# { name: "cement_19", lon: 31.72156, lat: 30.00699 },
# { name: "cement_20", lon: 30.90195, lat: 29.88331 },
# { name: "cement_21", lon: 30.89037, lat: 29.89328 },
# { name: "cement_22", lon: 30.86475, lat: 29.94431 },
# { name: "cement_23", lon: 30.88277, lat: 29.91311 },
# { name: "cement_24", lon: 30.92410, lat: 29.92849 },
# { name: "cement_25", lon: 30.88925, lat: 29.93126 },
# { name: "cement_26", lon: 30.91297, lat: 29.92560 },
# { name: "cement_27", lon: 30.90503, lat: 29.90133 },
# { name: "cement_28", lon: 30.86422, lat: 29.94251 },
# { name: "cement_29", lon: 30.89362, lat: 29.90350 },
# { name: "cement_30", lon: 30.91232, lat: 29.91913 },
# { name: "cement_31", lon: 30.92238, lat: 29.93308 },
# { name: "cement_32", lon: 30.89022, lat: 29.94841 },
# { name: "cement_33", lon: 30.92346, lat: 29.94345 },
# { name: "cement_34", lon: 30.91333, lat: 29.94939 },
# { name: "cement_35", lon: 30.92644, lat: 29.93806 },
# { name: "cement_36", lon: 30.90255, lat: 29.88953 },
# { name: "cement_37", lon: 30.94312, lat: 29.93434 },
# { name: "cement_38", lon: 30.94492, lat: 29.94961 },
# { name: "cement_39", lon: 30.94034, lat: 29.90058 },
# { name: "cement_40", lon: 30.87607, lat: 29.91709 },
# { name: "cement_41", lon: 30.93105, lat: 29.94373 },
# { name: "cement_42", lon: 30.87181, lat: 29.91438 },
# { name: "cement_43", lon: 31.44083, lat: 30.15340 },
# { name: "cement_44", lon: 31.35933, lat: 30.16616 },
# { name: "cement_45", lon: 31.40028, lat: 30.18061 },
# { name: "cement_46", lon: 31.39996, lat: 30.16244 },
# { name: "cement_47", lon: 31.35421, lat: 30.24893 },
# { name: "cement_48", lon: 31.36682, lat: 30.16972 },
# { name: "cement_49", lon: 31.41819, lat: 30.15220 },
# { name: "cement_50", lon: 31.40408, lat: 30.17497 },
# { name: "cement_51", lon: 31.41868, lat: 30.17532 },
# { name: "cement_52", lon: 31.39157, lat: 30.17611 },
# { name: "cement_53", lon: 31.44783, lat: 30.16074 },
# { name: "cement_54", lon: 31.35694, lat: 30.17337 },
# { name: "cement_55", lon: 31.43147, lat: 30.17385 },
# { name: "cement_56", lon: 31.43898, lat: 30.20823 },
# { name: "cement_57", lon: 31.42162, lat: 30.18947 },
# { name: "cement_58", lon: 31.42685, lat: 30.22541 },
# { name: "cement_59", lon: 31.37089, lat: 30.24945 },
# { name: "cement_60", lon: 31.35314, lat: 30.17947 },
# { name: "cement_61", lon: 31.40056, lat: 30.16342 },
# { name: "cement_62", lon: 31.41637, lat: 30.15738 },
# { name: "cement_63", lon: 31.44238, lat: 30.18359 },
# { name: "cement_64", lon: 31.37007, lat: 30.20657 },
# { name: "cement_65", lon: 31.36310, lat: 30.19116 },
# { name: "cement_66", lon: 31.37301, lat: 30.22148 },
# { name: "cement_67", lon: 31.35296, lat: 30.21177 },
# { name: "cement_68", lon: 31.44284, lat: 30.22667 },
# { name: "cement_69", lon: 31.37823, lat: 30.16103 },
# { name: "cement_70", lon: 31.44328, lat: 30.16585 },
# { name: "cement_71", lon: 31.40045, lat: 30.15439 },
# { name: "cement_72", lon: 31.35179, lat: 30.20214 },
# { name: "cement_73", lon: 31.42130, lat: 30.17962 },
# { name: "cement_74", lon: 31.31127, lat: 29.96685 },
# { name: "cement_75", lon: 31.30507, lat: 30.01087 },
# { name: "cement_76", lon: 31.25888, lat: 30.03996 },
# { name: "cement_77", lon: 31.34380, lat: 30.03345 },
# { name: "cement_78", lon: 31.29040, lat: 29.97065 },
# { name: "cement_79", lon: 31.33473, lat: 29.97809 },
# { name: "cement_80", lon: 31.33422, lat: 30.01781 },
# { name: "cement_81", lon: 31.33694, lat: 30.00190 },
# { name: "cement_82", lon: 31.33032, lat: 30.01303 },
# { name: "cement_83", lon: 31.28566, lat: 29.98231 },
# { name: "cement_84", lon: 31.27642, lat: 30.04529 },
# { name: "cement_85", lon: 31.34065, lat: 30.04373 },
# { name: "cement_86", lon: 31.26230, lat: 30.02499 },
# { name: "cement_87", lon: 31.33720, lat: 30.04964 },
# { name: "cement_88", lon: 31.34792, lat: 29.98802 },
# { name: "cement_89", lon: 31.28765, lat: 30.01502 },
# { name: "cement_90", lon: 31.28933, lat: 30.04587 },
# { name: "cement_91", lon: 31.29181, lat: 30.00182 },
# { name: "cement_92", lon: 31.25774, lat: 29.95267 },
# { name: "cement_93", lon: 31.29770, lat: 30.03717 },
# { name: "cement_94", lon: 31.34602, lat: 30.00757 },
# { name: "cement_95", lon: 31.27410, lat: 30.01278 },
# { name: "cement_96", lon: 31.28782, lat: 29.98147 },
# { name: "cement_97", lon: 31.33940, lat: 30.00256 },
# { name: "cement_98", lon: 31.25524, lat: 30.03101 },
# { name: "cement_99", lon: 31.29520, lat: 29.99398 },
# { name: "cement_100", lon: 31.29252, lat: 30.00878 },
# { name: "cement_101", lon: 31.32035, lat: 30.01216 },
# { name: "cement_102", lon: 31.30585, lat: 30.00628 },
# { name: "cement_103", lon: 31.26413, lat: 30.04840 },
# { name: "cement_104", lon: 31.28329, lat: 30.04270 },
# { name: "cement_105", lon: 31.25717, lat: 30.01068 },
# { name: "cement_106", lon: 31.31090, lat: 29.99331 },
# { name: "cement_107", lon: 31.32984, lat: 29.99879 },
# { name: "cement_108", lon: 31.33119, lat: 30.00558 },
# { name: "cement_109", lon: 31.27145, lat: 29.75713 },
# { name: "cement_110", lon: 31.28629, lat: 29.75091 },
# { name: "cement_111", lon: 31.29496, lat: 29.83350 },
# { name: "cement_112", lon: 31.34236, lat: 29.84893 },
# { name: "cement_113", lon: 31.29160, lat: 29.75809 },
# { name: "cement_114", lon: 31.30855, lat: 29.80698 },
# { name: "cement_115", lon: 31.32670, lat: 29.83635 },
# { name: "cement_116", lon: 31.28668, lat: 29.76851 },
# { name: "cement_117", lon: 31.26112, lat: 29.78398 },
# { name: "cement_118", lon: 31.29776, lat: 29.79908 },
# { name: "cement_119", lon: 31.29775, lat: 29.77597 },
# { name: "cement_120", lon: 31.31619, lat: 29.81194 },
# { name: "cement_121", lon: 31.30701, lat: 29.80501 },
# { name: "cement_122", lon: 31.25641, lat: 29.82549 },
# { name: "cement_123", lon: 31.31453, lat: 29.81060 },
# { name: "cement_124", lon: 31.32891, lat: 29.83545 },
# { name: "cement_125", lon: 31.27681, lat: 29.80360 },
# { name: "cement_126", lon: 31.25378, lat: 29.76875 },
# { name: "cement_127", lon: 31.26434, lat: 29.84389 },
# { name: "cement_128", lon: 31.30086, lat: 29.75605 },
# { name: "cement_129", lon: 31.32909, lat: 29.84961 },
# { name: "cement_130", lon: 31.28355, lat: 29.83025 },
# { name: "cement_131", lon: 31.27360, lat: 29.79771 },
# { name: "cement_132", lon: 31.26585, lat: 29.83640 },
# { name: "cement_133", lon: 31.29869, lat: 29.75498 },
# { name: "cement_134", lon: 31.31532, lat: 29.83370 },
# { name: "cement_135", lon: 31.32248, lat: 29.78496 },
# { name: "cement_136", lon: 31.27148, lat: 29.76061 },
# { name: "cement_137", lon: 31.27724, lat: 29.76986 },
# { name: "cement_138", lon: 31.29967, lat: 29.82470 },
# { name: "cement_139", lon: 31.30591, lat: 29.83912 },
# { name: 'cement_140', lon: 31.00680, lat: 30.78246 },

